[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haohanchen/POLI3148_2026Spring_TextAnalysis/blob/main/lecture/session_4_llm_text_analysis/session_4_llm_text_analysis.ipynb)

# Session 4: LLM-Powered Text Analysis. From API to Application

**POLI3148 Data Science in Politics and Public Administration**
The University of Hong Kong | Dr. Chen Haohan

---

In Sessions 1 to 3, we used what we can call **"Small" Language Models**. These are methods built from relatively small text datasets:

- **Rule-based / statistical:** tokenization, word frequencies, TF-IDF, dictionary-based sentiment (VADER)
- **Supervised learning:** ML classifiers trained on labeled examples
- **Unsupervised learning with strong assumptions:** LDA topic modeling

Each method required us to define features, build pipelines, and sometimes label training data by hand.

Now we take a fundamentally different approach. **Large Language Models (LLMs)** can perform many of the same tasks (sentiment analysis, named entity recognition, summarization, and more), but instead of writing feature engineering code, we describe what we want **in natural language**. The model does the rest.

In this session, we will:

1. Set up an LLM API connection
2. Use the LLM for **Sentiment Analysis** and compare with dictionary-based scores
3. Use the LLM for **Named Entity Extraction** and compare with spaCy-based labels
4. Use the LLM for **single-document summarization**, something traditional methods struggle with
5. Use the LLM for **aggregate summarization** with a topic tree, then compare against LDA from Session 2
6. Think about **cost**, **structured output**, and **hallucination risks**

**Data:** MoFA Press Conference Corpus (same dataset from Sessions 1 to 3).

## Setup

In [ ]:
!pip install -q openai pandas openpyxl networkx matplotlib

In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time

pd.set_option("display.max_colwidth", 200)

---
## Part 1: Setting Up the LLM API

There are two ways to use an LLM in your research workflow:

- **API (remote):** call a hosted model (GPT-4o, Claude, Gemini, Gemma...) over the internet. Frontier-quality models, no hardware needed, but each call costs tokens and your data leaves your machine.
- **Local (on your machine):** run an open-weight model (Llama, Gemma, Qwen...) on your own GPU/CPU. Free to run and fully private, but constrained by your hardware.

For this course we use the **API** route via **OpenRouter**, a unified gateway that routes to many model providers through a single OpenAI-compatible interface. You can switch between models just by changing one line of code.

For this session, we use **`google/gemma-4-26b-a4b-it`**, a small **paid** instruction-tuned model on OpenRouter. Earlier drafts of this notebook used a free-tier Gemma model, but free models on OpenRouter come with strict rate limits (requests per minute and daily caps) that make annotation at scale unreliable. Switching to a small paid model costs very little and removes the throttling.

**How an API call works:**
1. You send a **prompt** (system instructions + user input) to the API
2. The model processes it and sends back a **response**
3. You parse the response and use the results downstream

Each API call is billed in **tokens** (roughly 4 characters of English per token; see the [OpenAI Tokenizer](https://platform.openai.com/tokenizer) for an interactive demo). We will benchmark the per-call cost in Part 7.

In [ ]:
# Paste your OpenRouter API key here
OPENROUTER_API_KEY = "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"  # Replace with your actual key

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)
MODEL = "google/gemma-4-26b-a4b-it"  # small paid model; see Part 7 for cost

In [ ]:
def ask_llm(system_prompt, user_prompt, temperature=0.1, show_thinking=False):
    """Send a prompt to the LLM and return the response as a string.

    Parameters:
        system_prompt: Instructions that define the LLM's role and behavior
        user_prompt: The actual question or text to analyze
        temperature: Controls randomness (0 = deterministic, 1 = creative).
                     We use 0.1 for analysis tasks where we want consistent results.
        show_thinking: If True, ask the LLM to reason out loud BEFORE giving its
                     final answer. The reasoning is wrapped in <thinking>...</thinking>
                     tags so you can see *why* the model said what it said. Turn this
                     OFF when annotating at scale: the extra tokens add cost and
                     break structured-output parsing.
    """
    if show_thinking:
        system_prompt = (
            system_prompt
            + "\n\nBefore your final answer, write a brief reasoning paragraph "
              "wrapped in <thinking>...</thinking> tags. Then write your final "
              "answer immediately after, outside the tags."
        )
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API Error: {e}")
        return None

In [ ]:
# Test the API with a simple question.
# Set show_thinking=True so we can see the model's reasoning before its final answer.
result = ask_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="What is the capital of France? Answer in one sentence.",
    show_thinking=True,
)
print(result)

If you see a reasonable answer above, your API connection is working. If you see an error, double-check your API key.

---
## Part 2: Your First LLM Text Analysis

Let's load our MoFA data and send a real press conference question to the LLM for analysis.

In [ ]:
# Load the MoFA Press Conference corpus
df = pd.read_excel("https://github.com/haohanchen/POLI3148_2026Spring_TextAnalysis/raw/main/data/CMFA_PressCon_v6.xlsx")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

In [ ]:
# Pick a substantively interesting question. Let's find one about Taiwan
taiwan_rows = df[df["question"].str.contains("Taiwan", case=False, na=False)]
sample_row = taiwan_rows.iloc[0]

print(f"Date: {sample_row['date']}")
print(f"Spokesperson: {sample_row['spokesperson']}")
print(f"\n=== QUESTION ===")
print(sample_row["question"])

In [ ]:
# Send it to the LLM for open-ended analysis.
# Use show_thinking=True here too: this is exactly the kind of judgement task
# where seeing the reasoning helps you decide whether to trust the output.
analysis = ask_llm(
    system_prompt="You are a political analyst specializing in Chinese foreign policy.",
    user_prompt=f"Analyze this press conference question from China's Ministry of Foreign Affairs. "
                f"What is the topic? What is the tone? What is the journalist trying to learn?\n\n"
                f"Question: {sample_row['question']}",
    show_thinking=True,
)
print(analysis)

Notice what just happened: the LLM analyzed the text with **no feature engineering, no training data, no labeled examples**. Just a natural language instruction. This is the core advantage of LLM-based text analysis. Compare with the **traditional pipeline** from Sessions 1 to 3:

- *Traditional:* raw text → tokenize / lemmatize / remove stopwords → build document-term matrix → train or apply a rule-based / ML algorithm → results
- *LLM:* raw text → write prompt → call the LLM → results

LLM is simpler, but it costs money (or rate-limit budget) and is less transparent.

We also turned on `show_thinking=True` for this open-ended call. When you start using a new model on a new task, exposing the reasoning is the single most useful debugging tool you have. Once you trust the model on the task, **turn thinking off** for scale: the extra reasoning tokens cost money and get in the way of structured-output parsing.

Open-ended analysis like the one above is hard to use at scale. For systematic research, we need **structured output**. That means consistent labels we can put in a spreadsheet and analyze quantitatively. We will request **JSON** in the next task.

---
## Part 3: Task 1. Sentiment Annotation

In Session 3, we used **VADER** (a dictionary-based tool) to score sentiment. The dataset column `q_sentiment` contains those scores. Can an LLM do the same task? How do the results compare?

The key difference: VADER counts positive and negative words using a fixed dictionary. The LLM reads and *understands* the full sentence in context.

In [ ]:
# Define the sentiment classification prompt
# Following the prompt-engineering principles from the slides:
#   - Provide context (this is a press conference question)
#   - Be specific (state the task, the label set, and what each label means)
#   - Define the output format (JSON with named keys)
#   - Constrain the answer (return ONLY the JSON)
SENTIMENT_SYSTEM_PROMPT = """
You are a sentiment classifier for diplomatic press conference questions.

Conduct sentiment analysis on the question. Classify its sentiment as one of: positive, negative, or neutral.
- positive = expresses approval, praise, or optimism.
- negative = expresses criticism, accusation, or hostility.
- neutral = factual or procedural, no clear positive or negative stance.

Return your answer as JSON: {"sentiment": "...", "confidence": 0.0-1.0, "reasoning": "brief explanation"}
Return ONLY the JSON, no other text.
"""

In [ ]:
# Test on 3 individual questions with different sentiment levels.
# Pick one with low q_sentiment, one near zero, one with high q_sentiment.
test_cases = pd.concat([
    df.nsmallest(1, "q_sentiment"),                                    # most negative
    df.iloc[(df["q_sentiment"] - 0).abs().argsort()[:1]],              # most neutral
    df.nlargest(1, "q_sentiment"),                                     # most positive
]).reset_index(drop=True)

for i, row in test_cases.iterrows():
    label = ["NEGATIVE", "NEUTRAL", "POSITIVE"][i]
    print(f"=== Test {i+1}: VADER says {label} ===")
    print(f"VADER q_sentiment score: {row['q_sentiment']:.3f}")
    print(f"Question text:")
    print(row["question"])
    print()
    
    result = ask_llm(SENTIMENT_SYSTEM_PROMPT, row["question"])
    print(f"LLM response: {result}")
    print()
    time.sleep(1)  # Be respectful of rate limits

In [ ]:
for i, row in test_cases.iterrows():
    label = ["NEGATIVE", "NEUTRAL", "POSITIVE"][i]
    print(f"=== Test {i+1} ({label}, q_sentiment={row['q_sentiment']:.3f}) ===")
    print(f"Question: {str(row['question'])[:200]}...")
    print()
    
    result = ask_llm(SENTIMENT_SYSTEM_PROMPT, row["question"])
    print(f"LLM response: {result}")
    print()
    time.sleep(1)  # Be respectful of rate limits

In [ ]:
# Helper function to safely parse JSON from LLM responses
def parse_json_response(text):
    """Try to parse JSON from the LLM response. Returns dict or None."""
    if text is None:
        return None
    try:
        # Sometimes the LLM wraps JSON in markdown code blocks
        cleaned = text.strip()
        if cleaned.startswith("```"):
            # Remove markdown code fences
            lines = cleaned.split("\n")
            lines = [l for l in lines if not l.strip().startswith("```")]
            cleaned = "\n".join(lines)
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"  Warning: Could not parse JSON from response: {text[:100]}...")
        return None

In [ ]:
# Process 10 questions with varied sentiment scores
# Sample across the sentiment range for an interesting comparison
sentiment_sample = pd.concat([
    df.nsmallest(3, "q_sentiment"),
    df.iloc[(df["q_sentiment"] - 0).abs().argsort()[:4]],
    df.nlargest(3, "q_sentiment"),
]).drop_duplicates().head(10).reset_index(drop=True)

print(f"Processing {len(sentiment_sample)} questions through the LLM...\n")

results = []
for i, row in sentiment_sample.iterrows():
    print(f"  Processing question {i+1}/{len(sentiment_sample)}...")
    raw_response = ask_llm(SENTIMENT_SYSTEM_PROMPT, row["question"])
    parsed = parse_json_response(raw_response)
    
    results.append({
        "question": str(row["question"])[:100] + "...",
        "q_sentiment_score": round(row["q_sentiment"], 3),
        "llm_sentiment": parsed.get("sentiment", "PARSE_ERROR") if parsed else "PARSE_ERROR",
        "llm_confidence": parsed.get("confidence", None) if parsed else None,
        "llm_reasoning": parsed.get("reasoning", "") if parsed else "",
    })
    time.sleep(1)

results_df = pd.DataFrame(results)
print("\nDone!")

In [ ]:
# Compare LLM sentiment labels with the dataset's dictionary-based scores
display(results_df[["question", "q_sentiment_score", "llm_sentiment", "llm_confidence", "llm_reasoning"]])

**Discussion:**

- The `q_sentiment_score` column comes from VADER, a dictionary-based method. Positive scores mean more positive words; negative scores mean more negative words.
- The LLM provides a categorical label plus reasoning. It considers context and meaning, not just individual words.
- Where do they agree? Where do they disagree? Read the `llm_reasoning` column. Does the LLM's judgment seem more reasonable in some cases?

A common finding: dictionary methods can be fooled by negation ("not good" has "good" in it) or by domain-specific language. LLMs handle these cases better because they understand context.

### Improving with Few-Shot Examples

We can improve the LLM's performance by showing it a few labeled examples in the prompt. This is called **few-shot prompting**.

In [ ]:
# Few-shot sentiment prompt: we add 2 labeled examples so the LLM
# understands exactly what we expect. Same 3-level scheme as the zero-shot prompt.
SENTIMENT_FEWSHOT_PROMPT = """You are a sentiment classifier for diplomatic press conference questions.

Conduct sentiment analysis on the question. Classify its sentiment as one of: positive, negative, or neutral.
- positive = expresses approval, praise, or optimism.
- negative = expresses criticism, accusation, or hostility.
- neutral = factual or procedural, no clear positive or negative stance.

Here are two examples:

Example 1:
Text: "Reports say your government is deliberately covering up human rights abuses. How do you respond to these serious allegations?"
Output: {"sentiment": "negative", "confidence": 0.9, "reasoning": "Accusatory tone with strong negative framing (covering up, abuses, serious allegations)."}

Example 2:
Text: "Can you tell us about the outcomes of the recent trade agreement signing ceremony?"
Output: {"sentiment": "positive", "confidence": 0.8, "reasoning": "Cooperative event framed in approving terms (agreement, signing ceremony)."}

Now classify the following text.
Return your answer as JSON: {"sentiment": "...", "confidence": 0.0-1.0, "reasoning": "brief explanation"}
Return ONLY the JSON, no other text."""

In [ ]:
# Re-run on the same 10 questions with the few-shot prompt
print("Re-running sentiment analysis with few-shot prompt...\n")

fewshot_results = []
for i, row in sentiment_sample.iterrows():
    print(f"  Processing question {i+1}/{len(sentiment_sample)}...")
    raw_response = ask_llm(SENTIMENT_FEWSHOT_PROMPT, row["question"])
    parsed = parse_json_response(raw_response)
    
    fewshot_results.append({
        "question": str(row["question"])[:80] + "...",
        "q_sentiment_score": round(row["q_sentiment"], 3),
        "llm_zero_shot": results_df.iloc[i]["llm_sentiment"] if i < len(results_df) else "N/A",
        "llm_few_shot": parsed.get("sentiment", "PARSE_ERROR") if parsed else "PARSE_ERROR",
        "few_shot_reasoning": parsed.get("reasoning", "") if parsed else "",
    })
    time.sleep(1)

fewshot_df = pd.DataFrame(fewshot_results)
print("\nDone! Comparing zero-shot vs. few-shot results:")
display(fewshot_df)

Few-shot prompting often improves consistency because the model learns your labeling standards from the examples. In research applications, you would typically include 3 to 5 carefully chosen examples that cover edge cases.

---
## Part 4: Task 2. Named Entity Extraction

In Session 1, we used **spaCy** for Named Entity Recognition (NER). The dataset already contains pre-computed NER results in the columns `q_loc` (locations), `q_per` (persons), and `q_org` (organizations), separated by semicolons.

Let's see if the LLM can extract the same entities, and whether it finds anything spaCy missed.

In [ ]:
# Define the NER prompt
NER_SYSTEM_PROMPT = """Extract all named entities from the given text.
Categorize each entity as: PERSON, ORGANIZATION, LOCATION, or MISC.

Return your answer as JSON: {"entities": [{"text": "...", "type": "..."}, ...]}
If there are no named entities, return: {"entities": []}
Return ONLY the JSON, no other text."""

In [ ]:
# Select 10 questions that have at least some named entities in the dataset
ner_sample = df[
    (df["q_loc"] != "-") | (df["q_per"] != "-") | (df["q_org"] != "-")
].head(10).reset_index(drop=True)

print(f"Selected {len(ner_sample)} questions with pre-labeled entities.\n")

ner_results = []
for i, row in ner_sample.iterrows():
    print(f"  Processing question {i+1}/{len(ner_sample)}...")
    raw_response = ask_llm(NER_SYSTEM_PROMPT, row["question"])
    parsed = parse_json_response(raw_response)
    
    # Extract LLM entities by type
    llm_entities = parsed.get("entities", []) if parsed else []
    llm_locations = [e["text"] for e in llm_entities if e.get("type") == "LOCATION"]
    llm_persons = [e["text"] for e in llm_entities if e.get("type") == "PERSON"]
    llm_orgs = [e["text"] for e in llm_entities if e.get("type") == "ORGANIZATION"]
    
    ner_results.append({
        "question": str(row["question"])[:80] + "...",
        "dataset_locations": row["q_loc"],
        "llm_locations": "; ".join(llm_locations) if llm_locations else "-",
        "dataset_persons": row["q_per"],
        "llm_persons": "; ".join(llm_persons) if llm_persons else "-",
        "dataset_orgs": row["q_org"],
        "llm_orgs": "; ".join(llm_orgs) if llm_orgs else "-",
    })
    time.sleep(1)

ner_df = pd.DataFrame(ner_results)
print("\nDone!")

In [ ]:
# Compare: Dataset NER vs. LLM NER. Locations
print("=== LOCATIONS: Dataset vs. LLM ===\n")
display(ner_df[["question", "dataset_locations", "llm_locations"]])

print("\n=== PERSONS: Dataset vs. LLM ===\n")
display(ner_df[["question", "dataset_persons", "llm_persons"]])

print("\n=== ORGANIZATIONS: Dataset vs. LLM ===\n")
display(ner_df[["question", "dataset_orgs", "llm_orgs"]])

**Discussion:**

- How well does the LLM match the pre-labeled entities? Look for cases where they agree and where they differ.
- The LLM may find entities that spaCy missed (especially informal references to countries or people). It may also hallucinate entities that are not actually in the text. We will discuss this risk in Part 7.
- Notice that the LLM can also categorize entities in ways you specify. If you wanted a category like "GEOPOLITICAL_REGION" instead of "LOCATION", you could simply change the prompt.

---
## Part 5: Task 3. Summarization

This is where LLMs truly shine compared to traditional methods. Summarizing a Q&A exchange into a single sentence requires *understanding* the content, something bag-of-words models and word frequency counts cannot do.

In [ ]:
# Define the summarization prompt
SUMMARY_SYSTEM_PROMPT = """You are a research assistant summarizing diplomatic press conference exchanges.
Given a question and answer from a press conference, write a single-sentence summary 
that captures the key topic, the question's intent, and the main point of the answer.

Keep the summary under 30 words. Be factual and neutral.
Return ONLY the summary sentence, nothing else."""

In [ ]:
# Summarize 5 Q&A pairs
summary_sample = df.dropna(subset=["question", "answer"]).sample(n=5, random_state=42).reset_index(drop=True)

print("Generating summaries for 5 Q&A exchanges...\n")

summary_results = []
for i, row in summary_sample.iterrows():
    qa_text = f"QUESTION: {row['question']}\n\nANSWER: {row['answer']}"
        
    print(f"  Processing exchange {i+1}/5...")
    summary = ask_llm(SUMMARY_SYSTEM_PROMPT, qa_text)
    
    summary_results.append({
        "question": str(row["question"])[:120] + "...",
        "answer": str(row["answer"])[:120] + "...",
        "llm_summary": summary,
    })
    time.sleep(1)

summary_df = pd.DataFrame(summary_results)
print("\nDone!")

In [ ]:
# Display the original exchanges alongside their summaries
for i, row in summary_df.iterrows():
    print(f"=== Exchange {i+1} ===")
    print(f"Q: {row['question']}")
    print(f"A: {row['answer']}")
    print(f"SUMMARY: {row['llm_summary']}")
    print()

---
## Part 6: Task 4. Aggregate Summarization with a Topic Tree

Single-Q&A summarization is useful, but research often asks a different question: **what were the dominant themes across a whole set of documents?** In Session 2, we used **LDA topic modeling** to answer this. It gave us a flat list of topics, each represented by a word distribution.

An LLM can do something similar, but richer: we can ask it to produce a **hierarchical topic tree** with top-level themes, sub-topics within each, and specific issues under each sub-topic. This gives structured, human-readable output that LDA cannot produce.

**Benchmark:** compare the LLM's topic tree against the LDA topics from Session 2. Does the LLM identify the same broad themes? Does it surface nuances LDA missed?

**Data:** we'll feed the LLM a **one-month slice** of MoFA questions and ask for a topic tree of that month.

In [ ]:
# Pick one month of MoFA questions as our aggregate-summarization target
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year_month"] = df["date"].dt.to_period("M").astype(str)

TARGET_MONTH = "2022-08"  # August 2022, the Pelosi visit to Taiwan, expected to be topic-dense
month_slice = df[df["year_month"] == TARGET_MONTH].dropna(subset=["question"]).reset_index(drop=True)

print(f"Month: {TARGET_MONTH}")
print(f"Number of questions: {len(month_slice)}")

# Concatenate all questions into one block of text (numbered for traceability).
# If the month is too large for the context window, sample a representative subset.
MAX_QUESTIONS = 60
if len(month_slice) > MAX_QUESTIONS:
    month_slice = month_slice.sample(n=MAX_QUESTIONS, random_state=42).reset_index(drop=True)
    print(f"(Sampled {MAX_QUESTIONS} questions to stay within context window)")

questions_block = "\n".join(
    f"[{i+1}] {str(q)[:400]}" for i, q in enumerate(month_slice["question"])
)
print(f"Total character length of block: {len(questions_block):,}")

In [ ]:
# Topic tree prompt: ask the LLM to return nested JSON
TOPIC_TREE_SYSTEM_PROMPT = """You are a research assistant analyzing a set of diplomatic press conference questions from one month.

Your task: identify the hierarchical topic structure of the questions. Produce a **topic tree** with 3 to 6 top-level themes. Under each theme, list 2 to 5 sub-topics. Under each sub-topic, list 1 to 3 specific issues (with brief descriptors).

Return your answer as JSON with this exact structure:
{
  "month": "YYYY-MM",
  "topics": [
    {
      "topic": "Top-level theme name",
      "subtopics": [
        {
          "topic": "Sub-topic name",
          "issues": ["specific issue 1", "specific issue 2"]
        }
      ]
    }
  ]
}

Return ONLY the JSON, no other text."""

user_prompt = f"Month: {TARGET_MONTH}\n\nQuestions:\n{questions_block}"

raw_tree_response = ask_llm(TOPIC_TREE_SYSTEM_PROMPT, user_prompt, temperature=0.2)
topic_tree = parse_json_response(raw_tree_response)

if topic_tree:
    print(f"Topic tree generated for {topic_tree.get('month', TARGET_MONTH)}")
    print(f"Number of top-level topics: {len(topic_tree.get('topics', []))}")
else:
    print("Failed to parse topic tree. Raw response:")
    print(raw_tree_response)

In [ ]:
# Visualize the topic tree as an indented text tree
def print_topic_tree(tree):
    if not tree:
        return
    print(f"Topic tree for {tree.get('month', '')}")
    print("=" * 60)
    for top in tree.get("topics", []):
        print(f"\n* {top.get('topic', '(unnamed)')}")
        for sub in top.get("subtopics", []):
            print(f"    > {sub.get('topic', '(unnamed)')}")
            for issue in sub.get("issues", []):
                print(f"        . {issue}")

print_topic_tree(topic_tree)

In [ ]:
# Graphical visualization using networkx, drawn left-to-right
# (root on the left, top-level themes in the middle, sub-topics on the right).
import networkx as nx
import matplotlib.pyplot as plt

def build_tree_graph(tree):
    G = nx.DiGraph()
    root = f"{tree.get('month', '')} press conferences"
    G.add_node(root, level=0, label=root)
    for top in tree.get("topics", []):
        top_name = top.get("topic", "(unnamed)")
        top_key = f"L1::{top_name}"
        G.add_node(top_key, level=1, label=top_name)
        G.add_edge(root, top_key)
        for sub in top.get("subtopics", []):
            sub_name = sub.get("topic", "(unnamed)")
            # Make sub-keys unique in case the same name appears under multiple themes
            sub_key = f"L2::{top_name}::{sub_name}"
            G.add_node(sub_key, level=2, label=sub_name)
            G.add_edge(top_key, sub_key)
    return G, root

def hierarchy_pos_lr(G, root, height=1.0, horiz_gap=1.0,
                     horiz_loc=0.0, ycenter=0.5, pos=None, parent=None):
    """Recursive tree layout that grows LEFT to RIGHT.
    Root is placed at (horiz_loc, ycenter); each child is placed one column
    to the right (horiz_loc + horiz_gap), with siblings stacked vertically.
    """
    if pos is None:
        pos = {root: (horiz_loc, ycenter)}
    else:
        pos[root] = (horiz_loc, ycenter)
    children = [n for n in G.successors(root) if n != parent]
    if children:
        dy = height / len(children)
        nexty = ycenter - height / 2 - dy / 2
        for child in children:
            nexty += dy
            pos = hierarchy_pos_lr(G, child, height=dy, horiz_gap=horiz_gap,
                                   horiz_loc=horiz_loc + horiz_gap, ycenter=nexty,
                                   pos=pos, parent=root)
    return pos

if topic_tree:
    G, root = build_tree_graph(topic_tree)
    pos = hierarchy_pos_lr(G, root)
    
    labels = {n: G.nodes[n].get("label", n) for n in G.nodes}
    node_colors = ["#2D3748" if G.nodes[n]["level"] == 0
                   else "#48BB78" if G.nodes[n]["level"] == 1
                   else "#A0AEC0" for n in G.nodes]
    font_colors = "white"  # only used for the root; per-node coloring set below
    
    fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(G.nodes))))
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#CBD5E0", arrows=False)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=700, edgecolors="white", linewidths=1.5)
    
    # Place text labels to the RIGHT of each node so they don't overlap the dot.
    for n, (x, y) in pos.items():
        ax.text(x + 0.04, y, labels[n], fontsize=9,
                va="center", ha="left",
                color="#1A202C")
    
    ax.set_xlim(-0.1, max(p[0] for p in pos.values()) + 1.2)
    ax.set_axis_off()
    ax.set_title(f"LLM-generated topic tree: {TARGET_MONTH}", fontsize=13, loc="left")
    plt.tight_layout()
    plt.show()

**Discussion:**

- Compare this topic tree with the LDA topics you would get from Session 2's pipeline on the same month. Where do they align? Where do they diverge?
- **LDA** gives word distributions (e.g., `taiwan, visit, strait, pelosi, ...`). You have to *interpret* what the topic is.
- **LLM** gives named themes with structure, but it is also more opinionated. The LLM decides what counts as a theme based on its prior knowledge, not just the corpus.
- **Tradeoff:** LDA is reproducible and data-driven; LLM output is rich and human-readable but sensitive to prompt wording.
- For research: use both. Let LDA surface what the corpus emphasizes statistically; let the LLM organize and label those emphases into a readable hierarchy.

**Discussion:**

- Does each summary accurately capture the main point of the exchange?
- What information does the summary preserve? What does it lose?
- Summarization is a task where traditional text methods (word frequencies, topic models) simply cannot compete. You cannot generate fluent, accurate summaries by counting words.
- Think about how useful this would be at scale: imagine creating a one-sentence summary for each of the 35,000 exchanges, then filtering or searching through those summaries.

---
## Part 7: Structured Output and Cost Awareness

### Why Structured Output Matters

For research, we need outputs that are **consistent and machine-readable**. That is why every prompt above asks for **JSON** with a specific schema. JSON ([JavaScript Object Notation](https://www.json.org/)) is a standard format for structured data: key-value pairs, lists, and nested objects. Python reads JSON straight into a dict.

Structured output is what unlocks the *scale* benefits of LLMs:

- **Batch processing:** run the same prompt over thousands of documents and collect the results into a table
- **Automatic evaluation:** compare LLM output to a ground-truth column (precision, recall, F1)
- **Data pipelines:** LLM output becomes the input to the next step (a chart, a regression, a report)

Key practices:

- Always define the exact JSON structure you want in the system prompt
- Include the instruction "Return ONLY the JSON, no other text"
- Use `parse_json_response()` with error handling. The LLM will occasionally return malformed JSON.
- Use low temperature (0.0 to 0.2) for annotation tasks to reduce randomness

### Cost Awareness

We chose **`google/gemma-4-26b-a4b-it`** because it is small, fast, and very cheap per token. For text-annotation work like ours, you almost never need a frontier model: small "lite" models in the same class (Gemma lite, DeepSeek Flash, Gemini Flash) are usually enough.

Below we benchmark the cost of running our sentiment task across 100, 1k, 5k, and 35k MoFA questions on three comparable lite models. Pricing is in **USD per 1 million tokens** (input / output). Always check OpenRouter for current rates before scaling up.

In [ ]:
# Estimate token counts and costs across a few comparable lite models

# Average question length in characters
avg_question_chars = df["question"].dropna().str.len().mean()
avg_question_tokens = avg_question_chars / 4  # rough estimate: 1 token ~ 4 chars

# The system prompt for sentiment analysis
system_prompt_tokens = len(SENTIMENT_SYSTEM_PROMPT) / 4

# Estimated output: ~50 tokens for a JSON sentiment response
output_tokens_per_call = 50

# Total tokens per API call
input_tokens_per_call = system_prompt_tokens + avg_question_tokens
total_tokens_per_call = input_tokens_per_call + output_tokens_per_call

print(f"Average question length: {avg_question_chars:.0f} characters (~{avg_question_tokens:.0f} tokens)")
print(f"System prompt length: {len(SENTIMENT_SYSTEM_PROMPT)} characters (~{system_prompt_tokens:.0f} tokens)")
print(f"Estimated tokens per API call: ~{total_tokens_per_call:.0f} (input: {input_tokens_per_call:.0f}, output: {output_tokens_per_call})")
print()

# Compare a small set of "lite" models suitable for text annotation.
# Prices below are approximate USD per 1M tokens (input, output) on OpenRouter.
# Always verify current pricing at https://openrouter.ai/ before scaling up.
price_tiers = {
    "google/gemma-4-26b-a4b-it (chosen)": (0.10, 0.20),
    "deepseek/deepseek-v4-flash":         (0.27, 1.10),
    "google/gemini-2.5-flash":            (0.30, 2.50),
}

print("Estimated cost for sentiment annotation across the MoFA corpus:")
header = f"{'Scale':>22} | " + " | ".join(f"{name:>36}" for name in price_tiers)
print(header)
print("=" * len(header))
for n_questions in [100, 1000, 5000, 35000]:
    cells = [f"{n_questions:>6,} questions"]
    for name, (in_price, out_price) in price_tiers.items():
        input_cost = (input_tokens_per_call * n_questions / 1_000_000) * in_price
        output_cost = (output_tokens_per_call * n_questions / 1_000_000) * out_price
        total_cost = input_cost + output_cost
        cells.append(f"${total_cost:>7.3f}")
    print(f"{cells[0]:>22} | " + " | ".join(f"{c:>36}" for c in cells[1:]))

Key practices regardless of which model you use:

- **Work with subsets first.** Debug your prompts on 10 to 50 examples before scaling up.
- **Keep prompts concise.** Every extra word is multiplied by thousands of API calls.
- **Use the right model for the task.** A cheaper model may work just fine for simple classification.

---
## Part 8: Hallucination Warning

LLMs are powerful, but they have a critical weakness: they can generate **plausible-sounding but incorrect information**. This is called **hallucination**.

In our tasks today, hallucination can appear in several ways:

1. **Named Entity Recognition:** The LLM might "find" entities that are not actually in the text. For example, if a question mentions "the situation in the strait," the LLM might output "Taiwan Strait" even though those exact words were not used. Is that a correct extraction or a hallucination? It depends on your research question.

2. **Sentiment Analysis:** The LLM might be influenced by its training data rather than the actual text. If it "knows" that questions about human rights at MoFA press conferences tend to be negative, it might label a neutral question about human rights as negative.

3. **Summarization:** The LLM might add information that was not in the original text, or subtly shift the emphasis in a way that changes the meaning.

4. **Topic trees:** The LLM may invent themes that sound plausible but do not actually appear in the questions. Always verify a topic-tree branch against specific source questions before citing it.

### Mitigation Strategies

- **Use structured output** (JSON with specific fields) to constrain what the model can return
- **Use low temperature** (0.0 to 0.2) for annotation tasks to reduce creative improvisation
- **Compare with traditional methods** as a sanity check. If VADER and the LLM strongly disagree, investigate why.
- **Spot-check outputs manually.** Read a random sample of LLM annotations and verify them against the source text.
- **Never trust LLM output blindly.** Treat it as a first draft that needs human review, especially for high-stakes research.

---
## Your Turn

### Exercise

Write a custom prompt to classify MoFA questions into one of these topic categories:
- `territorial_disputes`
- `trade_and_economics`
- `human_rights`
- `state_visits_and_diplomacy`
- `military_and_security`
- `other`

Test on 5 questions and evaluate the results. Are the classifications reasonable? Which questions are hardest to classify?

In [ ]:
# YOUR CODE HERE: Define your topic classification prompt and test it

# Starter code:
TOPIC_SYSTEM_PROMPT = """Your prompt here..."""

# Pick 5 questions
exercise_sample = df.sample(n=5, random_state=123).reset_index(drop=True)

# Loop through and classify
for i, row in exercise_sample.iterrows():
    # result = ask_llm(TOPIC_SYSTEM_PROMPT, row["question"])
    # print(result)
    # time.sleep(1)
    pass

### Vibe Coding Challenge

Take the following prompt and give it to an LLM coding assistant (e.g., ChatGPT, Claude, or GitHub Copilot):

> "Modify this notebook to annotate 50 MoFA questions with both sentiment and topic labels in a single API call. Save the results to a CSV file for later analysis."

See what code it generates. Does it work? What would you change?